## 🚀 LLM Gateway Explained — Build One With LiteLLM + LangChain

### What is an LLM Gateway?
Think of an LLM Gateway as a smart middleware layer that sits between your application and multiple LLM providers (OpenAI, Anthropic, Google, Groq, Cohere, local models, etc.).

                    ┌─────────────────────────────┐
                    │       Your Application      │
                    │  (Chatbot, RAG, Agent, etc) │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │       LLM GATEWAY           │
                    │  • Routing                  │
                    │  • Fallbacks                │
                    │  • Caching                  │
                    │  • Rate Limiting            │
                    │  • Cost Tracking            │
                    │  • Observability            │
                    └──────┬─────┬─────┬─────┬────┘
                           │     │     │     │
                           ▼     ▼     ▼     ▼
                        OpenAI Claude Gemini Groq

### Without a Gateway (The Pain 😩)
- Different SDKs and APIs for every provider
- No fallback if one provider goes down
- No central place to track costs
- Hard to switch models without rewriting code
- No caching → paying twice for the same query

### With a Gateway (The Joy 😎)
- One unified API for 100+ providers
- Automatic fallbacks if a provider fails
- Centralized logging, cost tracking, rate limiting
- Swap models with a config change, no code rewrite
- Cache repeated queries → save money

In [1]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

#now import litellm normally
from litellm import completion

In [2]:
import litellm
litellm.suppress_debug_info = True

In [3]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [4]:
# Load API keys from a .env file
# Create a .env file in the same folder with:
# OPENAI_API_KEY=sk-...
# HUGGINGFACEHUB_API_TOKEN
# GROQ_API_KEY=gsk_...

import os
from dotenv import load_dotenv
load_dotenv()

# Quick check
print("OpenAI key loaded:    ", "✅" if os.getenv("OPENAI_API_KEY") else "❌")
print("google key loaded: ", "✅" if os.getenv("GOOGLE_API_KEY") else "❌")
print("Groq key loaded:      ", "✅" if os.getenv("GROQ_API_KEY") else "❌")

OpenAI key loaded:     ✅
google key loaded:  ✅
Groq key loaded:       ✅


In [5]:
from litellm import completion

# some code,different providers - just change the model strings
#call groq 

response_groq = completion(
    model = 'groq/openai/gpt-oss-20b',
    messages = [
        {"role":"user","content":"explain RAG in one sentence"}
    ]
)
print("groq:", response_groq.choices[0].message.content)

response_goolge = completion(
    model="gemini/gemini-3.5-flash-lite",
    messages = [
        {"role":"user","content":"explain RAG in one sentence"}
    ]
)
print("google:", response_goolge.choices[0].message.content)

groq: RAG (Retrieval‑Augmented Generation) is a generative‑AI technique that first fetches relevant passages from an external knowledge base and then conditions a language model’s output on those retrieved snippets to produce more accurate, context‑aware responses.
google: Retrieval-Augmented Generation (RAG) is a technique that improves AI responses by first searching an external database for relevant facts and then using that information to generate a more accurate, up-to-date answer.


In [6]:
from litellm import completion

prompt = "Explain RAG in one sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("🔵 OpenAI",     "gpt-4o-mini"),
    ("🟢 Groq",       "groq/openai/gpt-oss-20b"),
    ("🟣 Anthropic",  "claude-3-5-haiku-20241022"),
    ("🟡 Gemini",     "gemini/gemini-3.5-flash-lite"),
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:800]}")
    except Exception as e:
        print(f"{label:<15}: ❌ {type(e).__name__}")

🔵 OpenAI       : ❌ RateLimitError
🟢 Groq         : RAG (Retrieval‑Augmented Generation) is a framework that first retrieves relevant documents from a knowledge base and then feeds them into a generative model so it can produce more informed, context‑aware responses.
🟣 Anthropic    : ❌ BadRequestError
🟡 Gemini       : Retrieval-Augmented Generation (RAG) is a technique that improves AI responses by first searching an external database for relevant facts and then using that information to generate a more accurate, up-to-date answer.


# automatic fallback

In [7]:
from litellm import completion

#define a fallback chain

response = completion(
    model = "gpt-4o-mini",
    messages = [{"role":"user","content":"what is RAG?"}],
    fallbacks=[
        "groq/openai/gpt-oss-20b",
        "gemini/gemini-3.5-flash-lite"
    ]
)

print("response: ",response.choices[0].message.content[:200], "...")
print("\nwhich model actually answered?",response.model)

14:09:56 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gpt-4o-mini: litellm.RateLimitError: RateLimitError: OpenAIException - You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.
Traceback (most recent call last):
  File "c:\Users\preet\OneDrive\Desktop\Agentic_AI\.venv\Lib\site-packages\litellm\llms\openai\openai.py", line 888, in acompletion
    headers, response = await self.make_openai_chat_completion_request(
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
    )
    ^
  File "c:\Users\preet\OneDrive\Desktop\Agentic_AI\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 300, in async_wrapper
    result: Final = await func(*args, **kwargs)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\preet\OneDrive\Desktop\Agentic_AI\.venv\Lib\site-packages\litellm\llms\openai\openai.py", line 447, in make_op

response:  **RAG** most commonly refers to **Retrieval‑Augmented Generation** in the context of natural‑language processing (NLP) and AI. It’s a technique that combines a language model’s generative ability with ...

which model actually answered? openai/gpt-oss-20b


# cost tracking


In [8]:
from litellm import completion,completion_cost

response = completion(
    model = "gemini/gemini-3.5-flash-lite",
    messages = [{"role":"user","content":"waht is RAG and types of RAG"}]
)

#get cost 
cost = completion_cost(completion_response = response)

print("response:",response.choices[0].message.content)
print("\nInput tokens:",response.usage.prompt_tokens)
print("output tokens:",response.usage.completion_tokens)
print(f"cost:     ${cost:.8f}")

response: **RAG** stands for **Retrieval-Augmented Generation**. 

It is an AI technique used to improve the accuracy and reliability of Large Language Models (LLMs) by fetching facts from an external knowledge base before generating a response.

### The Problem RAG Solves
Standard LLMs (like GPT-4) only know what they were trained on. This leads to two main problems:
1. **Hallucinations:** They make up answers when they don't know the truth.
2. **Outdated Information:** They don't know about private company data, recent news, or events after their training cutoff.

### How RAG Works (In 3 Steps)
1. **Retrieve:** When a user asks a question, the system searches an external database (like company PDFs, a website, or a wiki) for relevant information.
2. **Augment:** The system combines the user's original question with the retrieved information into a single prompt.
3. **Generate:** The LLM reads the prompt containing the retrieved facts and generates an accurate, grounded response.

---

In [9]:
from litellm import completion, completion_cost

response = completion(
    model = "groq/openai/gpt-oss-20b",
    messages = [{"role":"user","content":"waht is RAG and types of RAG"}]
)

# get cost only when the model is mapped by LiteLLM
try:
    cost = completion_cost(completion_response = response)
    print("response:", response.choices[0].message.content)
    print("\nInput tokens:", response.usage.prompt_tokens)
    print("output tokens:", response.usage.completion_tokens)
    print(f"cost:     ${cost:.8f}")
except Exception as exc:
    print("response:", response.choices[0].message.content)
    print("\nInput tokens:", response.usage.prompt_tokens)
    print("output tokens:", response.usage.completion_tokens)
    print("cost tracking skipped:", exc)


response: ## 1.  What is **RAG**?

**RAG** stands for **Retrieval‑Augmented Generation**.  
It’s an architecture that lets a language model (LLM) answer questions or generate text by *pulling in* relevant external documents during inference (and sometimes during training).  The idea is to give the model access to a large, up‑to‑date knowledge base that can be queried on‑the‑fly, instead of having to embed all that knowledge inside the model’s parameters.

```
                ┌───────────────────┐
                │   Retrieval Layer │   (dense or sparse
                │   →  passage or    │   retrieval from a
                │   doc index)      │   vector store)
                └─────────┬─────────┘
                          │
                          ▼
                 ┌───────────────────────┐
                 │  Context‑Enriched     │
                 │   Language Model      │  (e.g., GPT‑3.5,
                 │   fed with retrieved  │   Llama‑2,
                 │   passages + pro

## caching 

In [10]:
import litellm

litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

litellm.cache = None

print("✅ litellm state reset - ready for clean caching done")

✅ litellm state reset - ready for clean caching done


In [11]:
import litellm
import time
from litellm import completion
from litellm.caching import Cache

#enable in memory caching(you can also use redis in production)
litellm.cache = Cache(type = "local")

prompt = "what does LLM stand for ? answer in on line"

#first call-actually hits openai
start = time.time()
r1 = completion(
    model = "gemini/gemini-3.5-flash-lite",
    messages=[{"role":"user","content":prompt}],
    caching = True
)
t1 = time.time() - start
print(f"❄️ first call(api):   {t1:.2f}s - {r1.choices[0].message.content}")

# second call served from cached , near-instant
start = time.time()
r2 = completion(
    model = "gemini/gemini-3.5-flash-lite",
    messages=[{"role":"user","content":prompt}],
    caching = True
)

t2 = time.time() - start

print(f"⚡ Second call (cache): {t2:.4f}s — {r2.choices[0].message.content}")
print(f"\n🚀 Speedup: {t1/t2:.1f}x faster, and ZERO cost on the second call!")

❄️ first call(api):   0.78s - LLM stands for Large Language Model.
⚡ Second call (cache): 0.0029s — LLM stands for Large Language Model.

🚀 Speedup: 272.8x faster, and ZERO cost on the second call!


# 🔀 Part 7: Smart Routing — The Right Model for the Right Job
### Why use one model for everything?
- Coding tasks → Claude Sonnet
- Cheap summaries → GPT-4o-mini
- Super fast replies → Groq Llama
- Complex reasoning → Claude Opus
- Use LiteLLM's Router to define routing rules:


In [12]:
import os
from litellm import Router

model_list = [
    {
        "model_name":"fast-cheap",
        "litellm_params":{
            "model":"gemini/gemini-3.5-flash-lite",
            "api_key":os.getenv("GOOGLE_API_KEY")
        }
    },
    {
        "model_name":"smart-coding",
        "litellm_params":{
            "model":"groq/openai/gpt-oss-20b",
            "api_key":os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name":"smart-coding",
        "litellm_params":{
            "model":"groq/qwen/qwen3.8-27b",
            "api_key":os.getenv("GROQ_API_KEY")
        }
    }
]

router = Router(model_list = model_list)

fast_response = router.completion(
    model = "fast-cheap",
    messages = [{"role":"user","content":"summarize:AI is changing software"}]
)

code_response = router.completion(
    model="smart-coding",
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}]
)

print("⚡ Fast/cheap (Groq): ", fast_response.choices[0].message.content[:150])
print("\n🧠 Smart/coding (GPT-4o):\n", code_response.choices[0].message.content[:300])

⚡ Fast/cheap (Groq):  AI is fundamentally transforming software development, making it faster, more automated, and accessible to non-programmers. 

Key changes include:
* *

🧠 Smart/coding (GPT-4o):
 Here’s a clean, single‑line implementation that uses Python’s slice syntax to reverse a string:

```python
def reverse_string(s: str) -> str:
    """
    Return a new string that is the reverse of the input string `s`.

    Parameters
    ----------
    s : str
        The string you want to reverse


## 🔁 Part 8: Load Balancing Across Multiple API Keys

In [13]:
from litellm import Router

#two deployments under the same alias
#a pool of smart model all equally capable just different provider

model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/qwen/qwen3.8-27b",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "qwen3.8-27b"}
    },
    
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "groq-gpt-oss-20b"}
    },
]
router = Router(
    model_list = model_list,
    routing_strategy = "simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-"*84)

for i in range(6):
    r = router.completion(
        model="gpt-pool",
        messages=[{"role": "user", "content": f"Say hello, request {i+1}"}]
    )
    # Pull out which deployment served this request
    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms   {answer}")

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------
#1        groq-gpt-oss-20b         637 ms   Hello! 🌟
#2        groq-gpt-oss-20b        1002 ms   Hello! Request 2.
#3        qwen3.8-27b              341 ms   Hello! How can I help you today?
#4        groq-gpt-oss-20b        1676 ms   Hello! Could you let me know what y
#5        groq-gpt-oss-20b        1023 ms   Hello! Could you let me know what y
#6        groq-gpt-oss-20b        1432 ms   Hello! 👋  

Could you let me know w


In [14]:
import os
from litellm import Router
from collections import Counter

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gemini/gemini-3.5-flash-lite",
                        "api_key": os.getenv("GOOGLE_API_KEY")},
     "model_info": {"id": "🔵 gemini"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/openai/gpt-oss-20b",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="least-busy"   # 👈 the magic
)

hits = Counter()
for i in range(8):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": f"Say 'OK' #{i}"}],
        max_tokens=5
    )
    hits[r._hidden_params.get("model_id", "?")] += 1
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")


print("\n🎯 Distribution:")
for k, v in hits.most_common():
    print(f"   {k}: {'█' * v} ({v})")


Request 1 → 🔵 gemini
Request 2 → 🔵 gemini
Request 3 → 🔵 gemini
Request 4 → 🔵 gemini
Request 5 → 🔵 gemini
Request 6 → 🔵 gemini
Request 7 → 🔵 gemini
Request 8 → 🔵 gemini

🎯 Distribution:
   🔵 gemini: ████████ (8)


In [15]:
import os
from litellm import Router
import time

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gemini/gemini-3.5-flash-lite",
                        "api_key": os.getenv("GOOGLE_API_KEY")},
     "model_info": {"id": "🔵 gemini/gemini-3.5-flash-lite"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/openai/gpt-oss-20b",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq gpt-oss-20b"}},
    
]

router = Router(
    model_list=model_list,
    routing_strategy="latency-based-routing"   # 👈 picks the fastest
)

# Send 10 requests and watch which deployments get picked over time
print(f"{'Req':<6}{'Deployment':<32}{'Latency':<10}")
print("-" * 50)

for i in range(10):
    start = time.time()
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Reply with exactly: OK"}],
        max_tokens=5
    )
    latency_ms = (time.time() - start) * 1000
    deployment = r._hidden_params.get("model_id", "?")
    print(f"#{i+1:<5}{deployment:<32}{latency_ms:>6.0f} ms")

Req   Deployment                      Latency   
--------------------------------------------------
#1    🔵 gemini/gemini-3.5-flash-lite    1231 ms
#2    🟢 Groq gpt-oss-20b                 512 ms
#3    🟢 Groq gpt-oss-20b                 454 ms
#4    🟢 Groq gpt-oss-20b                 550 ms
#5    🟢 Groq gpt-oss-20b                 457 ms
#6    🟢 Groq gpt-oss-20b                 537 ms
#7    🟢 Groq gpt-oss-20b                 473 ms
#8    🟢 Groq gpt-oss-20b                 486 ms
#9    🟢 Groq gpt-oss-20b                 354 ms
#10   🟢 Groq gpt-oss-20b                 438 ms


In [16]:
import os
from litellm import Router

# Different providers with very different price points
model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gemini/gemini-3.5-flash-lite",             # ~$2.50/M input tokens
                        "api_key": os.getenv("GOOGLE_API_KEY")},
     "model_info": {"id": "🔵 Gemini(Premium)"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/openai/gpt-oss-20b",        # ~$0.15/M input tokens
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🔵 Groq oss 20b (cheap)"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/qwen/qwen3.8-27b",   # ~$0.05/M
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq qwen (cheapest)"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"   # 👈 valid strategy
)

for i in range(5):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Hi"}],
        max_tokens=10
    )
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

Request 1 → 🔵 Groq oss 20b (cheap)
Request 2 → 🟢 Groq qwen (cheapest)
Request 3 → 🔵 Groq oss 20b (cheap)
Request 4 → 🟢 Groq qwen (cheapest)
Request 5 → 🔵 Gemini(Premium)


In [17]:
import litellm
from litellm import completion

# A simple in-memory log store
call_logs = []

def log_success(kwargs, completion_response, start_time, end_time):
    """Called automatically after every successful LLM call."""
    call_logs.append({
        "model": kwargs.get("model"),
        "prompt": kwargs["messages"][-1]["content"][:60],
        "input_tokens": completion_response.usage.prompt_tokens,
        "output_tokens": completion_response.usage.completion_tokens,
        "latency_sec": round((end_time - start_time).total_seconds(), 2),
        "cost_usd": kwargs.get("response_cost", 0),
        "user": kwargs.get("user", "anonymous")
    })

def log_failure(kwargs, completion_response, start_time, end_time):
    print("❌ Call failed:", kwargs.get("exception"))

# Register the callbacks
litellm.success_callback = [log_success]
litellm.failure_callback = [log_failure]

# Make a few tagged calls
for q, user in [
    ("What is RAG?", "krish"),
    ("Explain transformers.", "student_42"),
    ("What is fine-tuning?", "krish"),
]:
    completion(
        model="groq/openai/gpt-oss-120b",
        messages=[{"role": "user", "content": q}],
        user=user  # tag the call for attribution
    )

# Review the audit log
import json
print(json.dumps(call_logs, indent=2, default=str))

[]


In [18]:
!pip install -q langchain-litellm

### 🤖 Part 11: A Real Example — Multi-Provider LangChain Chain with Fallbacks
Let's combine everything: a LangChain chain that uses Claude as primary, with GPT and Groq as fallbacks — and logs every call.


In [24]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

#primary model
primary = ChatLiteLLM(model = "gpt-x")

#fallback
fallback_1 = ChatLiteLLM(model = "gpt-4o-mini",temperature = 0.2)
fallback_2 = ChatLiteLLM(model = "groq/openai/gpt-oss-20b",temperature = 0.2)

#langchain with_fallbacks() chain
robust_llm = primary.with_fallbacks([fallback_1,fallback_2])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI engineer. Always reply in JSON: {{\"answer\": ...}}"),
    ("user", "{question}")
])

chain = prompt | robust_llm | StrOutputParser()

result = chain.invoke({"question":"what are the top 3 benefits of an LLM Gateway"})
print(result)

❌ Call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=gpt-x
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
❌ Call failed: litellm.RateLimitError: RateLimitError: OpenAIException - You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.
{"answer":"1. Centralized control and governance – an LLM Gateway lets organizations enforce policies, monitor usage, and audit interactions across multiple models and vendors from a single point.\n2. Enhanced security and compliance – by routing all model traffic through a gateway, you can apply consistent authentication, encryption, and data‑handling rules, reducing the risk of data leaks and ensuring regulatory adherence.\n3. Simplified integration and scalability – the gateway abstra

mini end to end demo - smart router for a chatbot

In [31]:
import time
from litellm import completion , completion_cost

def classify_task(user_query:str) -> str:
    """
    cheap classifier -uses the fastest model to decide routing
    """
    cls = completion(
        model = "groq/openai/gpt-oss-120b",
        messages = [{
            "role":"user",
            "content":(
                f"Classify the following query into EXACTLY one word: "
                f"'code', 'summary', or 'general'. Query: {user_query}\n\nAnswer:"
            )
        }],
        max_tokens = 5
    )
    return cls.choices[0].message.content.strip().lower()

def call_with_fallbacks(model_chain,messages):
    """try each model in order, trturn the first one that succeeds"""
    last_error = None
    for model in model_chain:
        try:
            return completion(model = model,messages = messages)
        except Exception as e:
            print(f"   ⚠️  {model} failed ({type(e).__name__}), trying next...")
            last_error = e
            continue
    raise last_error

def smart_chat(user_query: str):
    """Routes to the right model based on task type, with fallbacks."""
    task = classify_task(user_query)

    # Each entry is a FULL chain: [primary, fallback1, fallback2, ...]
    # Every model name includes its provider prefix (groq/, anthropic/, etc.)
    routing = {
        "code":    ["gpt-4o",                     "gpt-4o-mini",   "groq/qwen/qwen3.8-27b"],
        "summary": ["gpt-4o-mini",                "groq/openai/gpt-oss-120b"],
        "general": ["groq/qwen/qwen3.8-27b", "gpt-4o-mini"],
    }
    model_chain = routing.get(task, routing["general"])

    start = time.time()
    response = call_with_fallbacks(
        model_chain=model_chain,
        messages=[{"role": "user", "content": user_query}]
    )
    latency = time.time() - start

    try:
        cost = completion_cost(completion_response=response)
        cost_str = f"${cost:.6f}"
    except Exception:
        cost_str = "n/a"

    return {
        "detected_task": task,
        "model_used":    response.model,
        "answer":        response.choices[0].message.content,
        "latency_sec":   round(latency, 2),
        "cost_usd":      cost_str
    }

# Try it on three very different queries
queries = [
    "Write a Python function to compute Fibonacci numbers.",
    "Summarize the importance of attention mechanism in 2 sentences.",
    "Tell me a fun fact about elephants."
]

for q in queries:
    print("=" * 70)
    print("❓ Q:", q)
    result = smart_chat(q)
    print(f"🏷️  Task:    {result['detected_task']}")
    print(f"🤖 Model:    {result['model_used']}")
    print(f"⏱️  Latency: {result['latency_sec']}s")
    print(f"💰 Cost:    {result['cost_usd']}")
    print(f"💬 Answer:  {result['answer'][:200]}...")

❓ Q: Write a Python function to compute Fibonacci numbers.
🏷️  Task:    
🤖 Model:    qwen/qwen3.8-27b
⏱️  Latency: 0.01s
💰 Cost:    n/a
💬 Answer:  # Fibonacci Number Calculator

Here are several approaches to compute Fibonacci numbers in Python, ranging from simple to optimized:

## 1. Basic Recursive (Educational – slow for large n)

```python
...
❓ Q: Summarize the importance of attention mechanism in 2 sentences.
🏷️  Task:    
🤖 Model:    qwen/qwen3.8-27b
⏱️  Latency: 0.84s
💰 Cost:    n/a
💬 Answer:  The attention mechanism allows neural networks to dynamically focus on the most relevant parts of the input data while ignoring irrelevant information, significantly enhancing the model's ability to c...
❓ Q: Tell me a fun fact about elephants.
🏷️  Task:    
🤖 Model:    qwen/qwen3.8-27b
⏱️  Latency: 0.84s
💰 Cost:    n/a
💬 Answer:  Elephants have **three stomachs**.

Like other large herbivores, elephants are equipped with a complex digestive system to break down tough plant material. The